# Adaptive Readout Scheduling: Early Stopping (Colab)

Notebook 09 builds on Notebook 08 by moving from **static readout subsets** to **adaptive / progressive readout**.

Core question:

```text
Can deterministic mod-style query schedules reach a target performance level
using fewer evaluated queries, with a reproducible stopping point?
```

Pipeline:

```text
many sparse test queries
    -> deterministic mod-style schedule
    -> progressive evaluation
    -> early stopping rule
    -> staged / adaptive inference
```

Guardrail: this notebook evaluates classical readout scheduling behavior. It does **not** claim QOS improvement, quantum advantage, or model accuracy improvement.


In [ ]:
# Clone your fork and move into repo root.
# If Colab says the folder already exists, restart runtime or comment these two lines after first run.
!git clone https://github.com/thinkthoughts/quantum-oracle-sketching-mod30.git
%cd quantum-oracle-sketching-mod30


In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modwheel import STANDARD_WHEELS

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

fig_dir = Path("figures")
data_dir = Path("data")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)


## 1. Load 20news and train baseline classifier

In [ ]:
RANDOM_STATE = 9423
N_RANDOM_TRIALS = 30

categories = [
    "comp.graphics",
    "rec.sport.baseball",
    "sci.space",
    "talk.politics.misc",
]

dataset = fetch_20newsgroups(
    subset="all",
    categories=categories,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=RANDOM_STATE,
)

texts = np.array(dataset.data, dtype=object)
y = np.array(dataset.target)
target_names = dataset.target_names

texts_train, texts_test, y_train, y_test = train_test_split(
    texts,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(max_features=12000, min_df=2, stop_words="english")
X_train = vectorizer.fit_transform(texts_train)
X_test = vectorizer.transform(texts_test)

clf = LinearSVC(random_state=RANDOM_STATE, dual="auto")
t0 = time.perf_counter()
clf.fit(X_train, y_train)
train_time = time.perf_counter() - t0

full_pred = clf.predict(X_test)
full_accuracy = accuracy_score(y_test, full_pred)
full_balanced_accuracy = balanced_accuracy_score(y_test, full_pred)

baseline = {
    "n_train": X_train.shape[0],
    "n_test": X_test.shape[0],
    "n_features": X_train.shape[1],
    "train_time_seconds": train_time,
    "full_accuracy": full_accuracy,
    "full_balanced_accuracy": full_balanced_accuracy,
}
baseline


## 2. Scheduling and progressive evaluation helpers

In [ ]:
def wheel_query_indices(n_queries, wheel_name="mod30"):
    wheel = STANDARD_WHEELS[wheel_name]
    residues = set(wheel.residues)
    return np.array([i for i in range(n_queries) if i % wheel.modulus in residues], dtype=int)


def random_query_schedule(n_queries, n_keep, seed):
    rng = np.random.default_rng(seed)
    return np.array(rng.choice(np.arange(n_queries), size=n_keep, replace=False), dtype=int)


def class_balance_l1_shift(y_subset, y_reference):
    n_classes = len(np.unique(y_reference))
    subset_counts = np.bincount(y_subset, minlength=n_classes)
    ref_counts = np.bincount(y_reference, minlength=n_classes)
    subset_frac = subset_counts / subset_counts.sum()
    ref_frac = ref_counts / ref_counts.sum()
    return float(np.sum(np.abs(subset_frac - ref_frac)))


def evaluate_indices(indices):
    pred = clf.predict(X_test[indices])
    yt = y_test[indices]
    return {
        "accuracy": accuracy_score(yt, pred),
        "balanced_accuracy": balanced_accuracy_score(yt, pred),
        "class_balance_l1_shift": class_balance_l1_shift(yt, y_test),
    }


def progressive_curve(schedule_indices, fractions):
    rows = []
    total_queries = X_test.shape[0]
    for frac in fractions:
        k = max(2, int(np.ceil(len(schedule_indices) * frac)))
        idx = schedule_indices[:k]
        t0 = time.perf_counter()
        metrics = evaluate_indices(idx)
        eval_time = time.perf_counter() - t0
        rows.append({
            "schedule_fraction": frac,
            "n_eval": len(idx),
            "fraction_of_all_queries": len(idx) / total_queries,
            "eval_time_seconds": eval_time,
            **metrics,
        })
    return pd.DataFrame(rows)


fractions = np.linspace(0.05, 1.0, 20)
wheel_query_indices(X_test.shape[0], "mod30")[:10], len(wheel_query_indices(X_test.shape[0], "mod30"))


## 3. Early stopping rules

We test two simple stopping rules:

1. **Target rule**: stop when balanced accuracy reaches a fraction of full-readout balanced accuracy.
2. **Stability rule**: stop when recent changes in balanced accuracy stay below a small tolerance.


In [ ]:
def stop_at_target(curve_df, target_fraction=0.95):
    target = target_fraction * full_balanced_accuracy
    for _, row in curve_df.iterrows():
        if row["balanced_accuracy"] >= target:
            return {
                "stop_reason": f"target_{target_fraction:.2f}",
                "target_balanced_accuracy": target,
                "stop_n_eval": int(row["n_eval"]),
                "stop_fraction_of_all_queries": float(row["fraction_of_all_queries"]),
                "stop_balanced_accuracy": float(row["balanced_accuracy"]),
                "stop_eval_time_seconds": float(row["eval_time_seconds"]),
            }
    last = curve_df.iloc[-1]
    return {
        "stop_reason": f"target_{target_fraction:.2f}_not_reached",
        "target_balanced_accuracy": target,
        "stop_n_eval": int(last["n_eval"]),
        "stop_fraction_of_all_queries": float(last["fraction_of_all_queries"]),
        "stop_balanced_accuracy": float(last["balanced_accuracy"]),
        "stop_eval_time_seconds": float(last["eval_time_seconds"]),
    }


def stop_at_stability(curve_df, window=3, tolerance=0.01):
    vals = curve_df["balanced_accuracy"].to_numpy()
    for i in range(window, len(vals)):
        recent = vals[i-window:i+1]
        if np.max(recent) - np.min(recent) <= tolerance:
            row = curve_df.iloc[i]
            return {
                "stop_reason": f"stable_w{window}_tol{tolerance}",
                "target_balanced_accuracy": np.nan,
                "stop_n_eval": int(row["n_eval"]),
                "stop_fraction_of_all_queries": float(row["fraction_of_all_queries"]),
                "stop_balanced_accuracy": float(row["balanced_accuracy"]),
                "stop_eval_time_seconds": float(row["eval_time_seconds"]),
            }
    last = curve_df.iloc[-1]
    return {
        "stop_reason": f"stable_w{window}_tol{tolerance}_not_reached",
        "target_balanced_accuracy": np.nan,
        "stop_n_eval": int(last["n_eval"]),
        "stop_fraction_of_all_queries": float(last["fraction_of_all_queries"]),
        "stop_balanced_accuracy": float(last["balanced_accuracy"]),
        "stop_eval_time_seconds": float(last["eval_time_seconds"]),
    }


## 4. Run deterministic and random progressive schedules

In [ ]:
curve_rows = []
stop_rows = []

for wheel_name in ["mod30", "mod210", "mod2310"]:
    det_schedule = wheel_query_indices(X_test.shape[0], wheel_name)
    det_curve = progressive_curve(det_schedule, fractions)
    det_curve["wheel"] = wheel_name
    det_curve["schedule_type"] = "deterministic"
    det_curve["trial"] = -1
    curve_rows.append(det_curve)

    for rule_name, stop_result in [
        ("target_95", stop_at_target(det_curve, target_fraction=0.95)),
        ("target_99", stop_at_target(det_curve, target_fraction=0.99)),
        ("stability", stop_at_stability(det_curve, window=3, tolerance=0.01)),
    ]:
        stop_rows.append({
            "wheel": wheel_name,
            "schedule_type": "deterministic",
            "trial": -1,
            "rule": rule_name,
            **stop_result,
        })

    n_keep = len(det_schedule)
    for trial in range(N_RANDOM_TRIALS):
        rand_schedule = random_query_schedule(
            X_test.shape[0],
            n_keep,
            seed=RANDOM_STATE + 9000 + trial * 1009 + len(wheel_name),
        )
        # Keep random schedule order as sampled; this represents a randomized readout order.
        rand_curve = progressive_curve(rand_schedule, fractions)
        rand_curve["wheel"] = wheel_name
        rand_curve["schedule_type"] = "random_matched"
        rand_curve["trial"] = trial
        curve_rows.append(rand_curve)

        for rule_name, stop_result in [
            ("target_95", stop_at_target(rand_curve, target_fraction=0.95)),
            ("target_99", stop_at_target(rand_curve, target_fraction=0.99)),
            ("stability", stop_at_stability(rand_curve, window=3, tolerance=0.01)),
        ]:
            stop_rows.append({
                "wheel": wheel_name,
                "schedule_type": "random_matched",
                "trial": trial,
                "rule": rule_name,
                **stop_result,
            })

curves_df = pd.concat(curve_rows, ignore_index=True)
stops_df = pd.DataFrame(stop_rows)

curves_csv = data_dir / "09_adaptive_readout_curves.csv"
stops_csv = data_dir / "09_adaptive_readout_stops.csv"
curves_df.to_csv(curves_csv, index=False)
stops_df.to_csv(stops_csv, index=False)

curves_df.head(), stops_df.head()


## 5. Summarize stopping behavior

In [ ]:
random_stop_summary = (
    stops_df[stops_df["schedule_type"] == "random_matched"]
    .groupby(["wheel", "rule"])
    .agg(
        random_stop_fraction_mean=("stop_fraction_of_all_queries", "mean"),
        random_stop_fraction_std=("stop_fraction_of_all_queries", "std"),
        random_stop_bal_acc_mean=("stop_balanced_accuracy", "mean"),
        random_stop_bal_acc_std=("stop_balanced_accuracy", "std"),
        random_stop_n_eval_mean=("stop_n_eval", "mean"),
        random_stop_n_eval_std=("stop_n_eval", "std"),
    )
    .reset_index()
)

det_stop = stops_df[stops_df["schedule_type"] == "deterministic"][[
    "wheel", "rule", "stop_fraction_of_all_queries", "stop_balanced_accuracy", "stop_n_eval", "stop_reason"
]].rename(columns={
    "stop_fraction_of_all_queries": "det_stop_fraction",
    "stop_balanced_accuracy": "det_stop_bal_acc",
    "stop_n_eval": "det_stop_n_eval",
})

stop_summary = det_stop.merge(random_stop_summary, on=["wheel", "rule"])
stop_summary["delta_stop_fraction_vs_random"] = stop_summary["det_stop_fraction"] - stop_summary["random_stop_fraction_mean"]
stop_summary["delta_stop_bal_acc_vs_random"] = stop_summary["det_stop_bal_acc"] - stop_summary["random_stop_bal_acc_mean"]
stop_summary["delta_stop_n_eval_vs_random"] = stop_summary["det_stop_n_eval"] - stop_summary["random_stop_n_eval_mean"]

summary_csv = data_dir / "09_adaptive_readout_stop_summary.csv"
stop_summary.to_csv(summary_csv, index=False)
stop_summary


## 6. Figure 09a — progressive curves

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 5.0))

for wheel_name in ["mod30", "mod210", "mod2310"]:
    det = curves_df[(curves_df["wheel"] == wheel_name) & (curves_df["schedule_type"] == "deterministic")]
    ax.plot(det["fraction_of_all_queries"], det["balanced_accuracy"], marker="o", linewidth=2, label=f"{wheel_name} deterministic")

ax.axhline(full_balanced_accuracy, linestyle="--", linewidth=1, label="full readout")
ax.axhline(0.95 * full_balanced_accuracy, linestyle=":", linewidth=1, label="95% target")
ax.set_title("Adaptive Readout: Progressive Deterministic Schedules")
ax.set_xlabel("Fraction of all test queries evaluated")
ax.set_ylabel("Balanced accuracy")
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.35)
ax.legend()
fig.tight_layout()

fig_a_svg = fig_dir / "figure_09a_adaptive_progressive_curves.svg"
fig_a_png = fig_dir / "figure_09a_adaptive_progressive_curves.png"
fig.savefig(fig_a_svg)
fig.savefig(fig_a_png, dpi=220)
plt.show()
fig_a_svg, fig_a_png


## 7. Figure 09b — stopping fraction vs random baseline

In [ ]:
rule_to_plot = "target_95"
plot_summary = stop_summary[stop_summary["rule"] == rule_to_plot].set_index("wheel").loc[["mod30", "mod210", "mod2310"]].reset_index()

x = np.arange(len(plot_summary))
width = 0.36

fig, ax = plt.subplots(figsize=(8.8, 5.0))
ax.bar(x - width/2, plot_summary["det_stop_fraction"], width, label="deterministic")
ax.bar(x + width/2, plot_summary["random_stop_fraction_mean"], width, yerr=plot_summary["random_stop_fraction_std"], label="random mean")
ax.set_title("Early Stop Fraction: Deterministic vs Random (95% Target)")
ax.set_xlabel("Schedule")
ax.set_ylabel("Fraction of all queries evaluated at stop")
ax.set_xticks(x)
ax.set_xticklabels(plot_summary["wheel"])
ax.grid(True, axis="y", alpha=0.35)
ax.legend()
fig.tight_layout()

fig_b_svg = fig_dir / "figure_09b_stop_fraction_vs_random.svg"
fig_b_png = fig_dir / "figure_09b_stop_fraction_vs_random.png"
fig.savefig(fig_b_svg)
fig.savefig(fig_b_png, dpi=220)
plt.show()
fig_b_svg, fig_b_png


## 8. Figure 09c — deterministic stopping delta

In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 5.0))
ax.axhline(0, linestyle="--", linewidth=1)
ax.bar(plot_summary["wheel"], plot_summary["delta_stop_fraction_vs_random"])
ax.set_title("Deterministic Stop Fraction Minus Random Mean (95% Target)")
ax.set_xlabel("Schedule")
ax.set_ylabel("Δ stop fraction")
ax.grid(True, axis="y", alpha=0.35)

for i, value in enumerate(plot_summary["delta_stop_fraction_vs_random"]):
    ax.text(i, value, f"{value:+.3f}", ha="center", va="bottom" if value >= 0 else "top")

fig.tight_layout()
fig_c_svg = fig_dir / "figure_09c_stop_fraction_delta.svg"
fig_c_png = fig_dir / "figure_09c_stop_fraction_delta.png"
fig.savefig(fig_c_svg)
fig.savefig(fig_c_png, dpi=220)
plt.show()
fig_c_svg, fig_c_png


## 9. Figure 09d — stability stopping rule

In [ ]:
rule_to_plot = "stability"
stab_summary = stop_summary[stop_summary["rule"] == rule_to_plot].set_index("wheel").loc[["mod30", "mod210", "mod2310"]].reset_index()

x = np.arange(len(stab_summary))
width = 0.36

fig, ax = plt.subplots(figsize=(8.8, 5.0))
ax.bar(x - width/2, stab_summary["det_stop_fraction"], width, label="deterministic")
ax.bar(x + width/2, stab_summary["random_stop_fraction_mean"], width, yerr=stab_summary["random_stop_fraction_std"], label="random mean")
ax.set_title("Early Stop Fraction: Stability Rule")
ax.set_xlabel("Schedule")
ax.set_ylabel("Fraction of all queries evaluated at stop")
ax.set_xticks(x)
ax.set_xticklabels(stab_summary["wheel"])
ax.grid(True, axis="y", alpha=0.35)
ax.legend()
fig.tight_layout()

fig_d_svg = fig_dir / "figure_09d_stability_stop_fraction.svg"
fig_d_png = fig_dir / "figure_09d_stability_stop_fraction.png"
fig.savefig(fig_d_svg)
fig.savefig(fig_d_png, dpi=220)
plt.show()
fig_d_svg, fig_d_png


## 10. Interpretation helper

In [ ]:
print("Full balanced accuracy:", full_balanced_accuracy)
print("\nTarget 95% of full:", 0.95 * full_balanced_accuracy)
print("\nStop summary:")
display(stop_summary)

for _, row in plot_summary.iterrows():
    print(
        f"{row['wheel']} target_95: det_stop_fraction={row['det_stop_fraction']:.3f}, "
        f"random_mean={row['random_stop_fraction_mean']:.3f}, "
        f"delta={row['delta_stop_fraction_vs_random']:+.3f}"
    )

print("""
Notebook 09 interpretation:

- Notebook 08 showed deterministic scheduling as a static readout-control pattern.
- Notebook 09 adds adaptive stopping: evaluate progressively and stop after reaching a target or stability criterion.
- If deterministic stop fractions are near random means, the value is reproducibility and structured stopping rather than accuracy improvement.
- If deterministic stop fractions are lower than random means, the schedule reaches the target with fewer queries in this experiment.
- If higher, deterministic scheduling still gives reproducible readout staging but may be less efficient than random for that criterion.

Guardrail:
This is an adaptive readout scheduling experiment, not a proof of QOS improvement, quantum advantage, or model improvement.
""")


## 11. Download outputs

In [ ]:
from google.colab import files

for path in [
    "data/09_adaptive_readout_curves.csv",
    "data/09_adaptive_readout_stops.csv",
    "data/09_adaptive_readout_stop_summary.csv",
    "figures/figure_09a_adaptive_progressive_curves.svg",
    "figures/figure_09b_stop_fraction_vs_random.svg",
    "figures/figure_09c_stop_fraction_delta.svg",
    "figures/figure_09d_stability_stop_fraction.svg",
]:
    files.download(path)
